# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# To display textual metadata summary
metadata_json = dataset.metadata.to_json()
print(f"{metadata_json['name']}: {metadata_json['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema organizes data as `record sets`, each identified by a unique `@id`. Each record set contains `fields`, also addressed by `@id`. Below, we enumerate the record sets and their fields present in the dataset.

In [ ]:
# List available record sets and their fields, by @id
record_sets = dataset.metadata.record_sets
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    print("  Fields:")
    for field in rs.get('field', []):
        # field is either string (@id) or dict
        if isinstance(field, dict):
            print(f"    - {field.get('@id', field)}")
        else:
            print(f"    - {field}")
    print()

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract all available record sets into DataFrames
import collections

rs_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in rs_ids:
    # For each record set, extract records
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0 and isinstance(records[0], collections.abc.Mapping):
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from RecordSet {record_set_id}.")
    elif len(records) > 0:
        print(f"Loaded {len(records)} records from RecordSet {record_set_id}, records are not dict-like: showing first 5.")
        print(records[:5])
    else:
        print(f"No records available for RecordSet {record_set_id}.")

# For demonstration, select the first available record set for further exploration
selected_rs = None
for rs_id in dataframes:
    selected_rs = rs_id
    break

if selected_rs is not None:
    print(f"Columns in selected RecordSet {selected_rs}:")
    print(dataframes[selected_rs].columns.tolist())
    display(dataframes[selected_rs].head())
else:
    print("No DataFrames loaded. Check for record set availability.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes.

We will demonstrate EDA on the selected record set and focus on numeric fields.

In [ ]:
import numpy as np

if selected_rs is None:
    print("No record set available for EDA.")
else:
    df = dataframes[selected_rs]
    # Try to find numeric columns
    numeric_candidate_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Numeric field candidates: {numeric_candidate_cols}")

    # If dataframe has numeric fields, pick the first for filtering/normalization
    if numeric_candidate_cols:
        numeric_field = numeric_candidate_cols[0]
        print(f"Using '{numeric_field}' for demonstration.")
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}): {len(filtered_df)} records")

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"],].head())

        # Attempt grouping by another column if available
        possible_group_fields = [col for col in df.columns if df[col].nunique() < len(df) and not np.issubdtype(df[col].dtype, np.number)]
        if possible_group_fields:
            group_field = possible_group_fields[0]
            print(f"Grouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(grouped_df.head())
        else:
            print("No suitable non-numeric group field found.")
    else:
        print("No numeric fields found in this record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, we demonstrate a histogram of the example numeric field or a boxplot by group if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

if selected_rs is not None and numeric_candidate_cols:
    # Histogram of the numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field], kde=True, bins=30)
    plt.title(f"Distribution of '{numeric_field}' in RecordSet {selected_rs}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If grouped field used above, show boxplot
    if 'group_field' in locals():
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"Boxplot of {numeric_field} grouped by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load and explore the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) Mass Spectrometry dataset, reviewed available record sets and performed basic exploratory data analysis, including numeric filtering and normalization, and visualized the results. For further study, investigate specific protein metrics, explore protein abundance variability, or link to UniProt using the accession fields.